In [11]:
import os, re, pickle
import pandas as pd
import numpy as np
from scipy.stats import zscore
import seaborn as sns
from scipy.stats import pearsonr
import torch.nn.functional as F
import tensorflow_hub as hub 
from transformers import GPT2Tokenizer, GPT2LMHeadModel
base_path = 'C:/Users\hchen\Dropbox\PycharmProjects/false_mem'
SAVE_PATH = base_path+'/GPT_output/results_arrays/'

def check_correct(t, c):
    return np.nan if np.isnan(t) else (t == 1 and c == 0)
# Vectorize the function to apply it element-wise
vectorize_correct = np.vectorize(check_correct,otypes=[object])

def check_false(t, c):
    return np.nan if np.isnan(t) else (t == 0 or c == 1)
# Vectorize the function to apply it element-wise
vectorize_false = np.vectorize(check_false,otypes=[object])

# Correlating three semantic measures with false memory (confabulation or conflicts)

### loading memory performances for all 4 stories

In [12]:
THRESHOLD = 20 #number of recalls
story_ids = ['pieman','eyespy','oregontrail','baseball']
axis = 0  # 0=event, 1=subject
arr_perc_false = []
arr_perc_correct = []
arr_perc_conflict = []
arr_perc_overlap = []
arr_perc_confab = []
arr_perc_recall = []
keep_ids = []
for story_id in story_ids:
    recall_path = base_path+'/GPT_output/memory_classification/%s' % story_id
    recall_paths = [recall_path + '/'+x
                    for x in os.listdir(recall_path) if 'prompt' not in x]
    story_segs = pd.read_excel('./A. script_prompting/story_segs.xlsx',
                           sheet_name='%s_segs' % story_id)['events'].to_list()

    match_path = 'C:/Users/hchen/Dropbox/PycharmProjects/false_mem/GPT_output/GPT_matching/%s' % story_id
    id_path = 'C:/Users/hchen\Dropbox\PycharmProjects\speech2text_topic\data\integrated/all_first_two.xlsx'
    id_sheet = pd.read_excel(id_path)
    confab = np.load(SAVE_PATH+'/confab_%s.npy'%(story_id))
    true = np.load(SAVE_PATH+'/conflict_%s.npy'%(story_id))
    # percentage recall
    percent_recall = []
    for i in range(len(story_segs)):
        percent_recall.append((len(recall_paths)-np.sum(np.logical_or(np.isnan(confab[:,i]),np.isnan(true[:,i]))))/len(recall_paths))

    correct = vectorize_correct(true, confab).astype(float)
    false = vectorize_false(true,confab).astype(float)
    num_correct = np.nansum(correct,axis=axis)
    perc_correct = np.nansum(correct, axis=axis)/(np.array(percent_recall)*len(recall_paths))
    num_conflict = np.nansum(true==0, axis=axis)
    perc_conflict = np.nansum(true==0, axis=axis)/(np.array(percent_recall)*len(recall_paths))
    num_confab = np.nansum(confab==1, axis=axis)
    perc_confab = np.nansum(confab==1, axis=axis)/(np.array(percent_recall)*len(recall_paths))
    num_false = np.nansum(false, axis=axis)
    perc_false = num_false/(np.array(percent_recall)*len(recall_paths))
    perc_overlap = perc_confab+perc_conflict-perc_false
    #
    """
    cleaning with threshold
    """
    keep_id = np.array(percent_recall)*len(recall_paths)>=THRESHOLD
    keep_ids.append(keep_id)
    print('removed event', len(percent_recall)-np.sum(keep_id))
    perc_false = perc_false[keep_id]
    perc_correct = perc_correct[keep_id]
    perc_conflict = perc_conflict[keep_id]
    perc_overlap = perc_overlap[keep_id]
    perc_confab = perc_confab[keep_id]
    percent_recall = np.array(percent_recall)[keep_id]
    
    arr_perc_false.append(perc_false)
    arr_perc_correct.append(perc_correct)
    arr_perc_conflict.append(perc_conflict)
    arr_perc_overlap.append(perc_overlap)
    arr_perc_confab.append(perc_confab)
    arr_perc_recall.append(percent_recall)

perc_false = np.concatenate(arr_perc_false)
perc_correct = np.concatenate(arr_perc_correct)
perc_conflict = np.concatenate(arr_perc_conflict)
perc_overlap = np.concatenate(arr_perc_overlap)
perc_confab = np.concatenate(arr_perc_confab)
perc_recall = np.concatenate(arr_perc_recall)
np.save(base_path+'/E. semantic_features/keep_ids', np.array(keep_ids, dtype=object), allow_pickle=True)


## 1. semantic centrality
models: USE, all-mpnet

In [18]:
from transformers import AutoTokenizer, AutoModel
from torch import Tensor
from sentence_transformers import SentenceTransformer
embedding_model = 'GIST'
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
def compute_cosine_similarity(embeddings):
    norm = np.linalg.norm(embeddings, axis=1)
    norm[norm == 0] = 1
    embeddings_normalized = embeddings / norm[:, np.newaxis]
    return np.dot(embeddings_normalized, embeddings_normalized.T)
def embed_model(embedding_model, txt):
    if embedding_model == 'USE':
        embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
        return embed(txt).numpy()
    elif embedding_model == 'GIST':
        model = SentenceTransformer("avsolatorio/GIST-Embedding-v0")
        return model.encode(txt)
    elif embedding_model == 'mpnet':
        model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        return model.encode(txt)
    elif embedding_model == 'bge':
        model = SentenceTransformer('BAAI/bge-large-zh-v1.5')
        return model.encode(txt)
    elif embedding_model == 'e5':
        tokenizer = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
        model = AutoModel.from_pretrained('intfloat/e5-base-v2')
        batch_dict = tokenizer(list(txt), max_length=512, padding=True, truncation=True, return_tensors='pt')
        outputs = model(**batch_dict)
        embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
        # F.normalize(embeddings, p=2, dim=1)
        return embeddings.detach().cpu().numpy()

In [19]:
centrality_arr = []
for j, story_id in enumerate(story_ids):
    story_segs = pd.read_excel('./A. script_prompting/story_segs.xlsx',
                           sheet_name='%s_segs' % story_id)['events'].to_list()
    story_segs = np.array(story_segs)[keep_ids[j]]
    embedding = embed_model(embedding_model, story_segs)
    cos_sim = compute_cosine_similarity(embedding)
    n = cos_sim.shape[0]
    # Add nodes with initial centrality values
    np.fill_diagonal(cos_sim, np.nan)
    centrality=np.nanmean(cos_sim, axis=0)
    
    norm_centrality = zscore(centrality)
    centrality_arr.extend(norm_centrality)
    
    np.save(base_path+'/E. semantic_features/centrality_%s_%s' % (embedding_model,story_id), centrality)

## 4. "Contextual" surprisal

In [49]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoTokenizer, AutoModelForCausalLM

def score_gpt_contextual(sentence, context, tokenizer, model, max_length=1024):    
    full_text = context + sentence
    tokenize_input = tokenizer.encode(full_text, return_tensors='pt')
    
    # Truncate the context if necessary
    if tokenize_input.size(1) > max_length:
        context_tokens = tokenizer.encode(context)
        sentence_tokens = tokenizer.encode(sentence)
        total_tokens = len(context_tokens) + len(sentence_tokens)
    
        if total_tokens > max_length:
            # Truncate the context
            truncated_context_tokens = context_tokens[-(max_length - len(sentence_tokens)):]
            tokenize_input = tokenizer.encode(truncated_context_tokens + sentence_tokens, return_tensors='pt')
        else:
            tokenize_input = tokenizer.encode(full_text, max_length=max_length, truncation=True, return_tensors='pt')
    
    # Tokenize only the sentence to create a mask
    sentence_tokens = tokenizer.encode(sentence, return_tensors='pt')
    
    # Create a mask for the loss calculation
    mask = torch.cat([torch.zeros(tokenize_input.size(1) - sentence_tokens.size(1)), torch.ones(sentence_tokens.size(1))])
    
    # Ensure tensor is on the same device as the model (e.g., GPU if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    tensor_input = tokenize_input.to(device)
    mask = mask.to(device)
    
    # Compute the loss
    with torch.no_grad():
        outputs = model(tensor_input, labels=tensor_input)
        loss = outputs[0]
    
    output = model(tensor_input, labels=tensor_input)
    logits = output.logits
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = tensor_input[..., 1:].contiguous()
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
    
    # Apply the mask to the loss
    masked_loss = loss * mask[1:]
    masked_loss = masked_loss.sum() / mask.sum()  # Average loss over the masked tokens
    return masked_loss.detach().cpu().numpy()
def score_pythia_contextual(sentence, context, tokenizer, model, max_length=1024):
    model.eval()

    # Tokenize without special tokens
    context_ids  = tokenizer.encode(context, add_special_tokens=False)
    sentence_ids = tokenizer.encode(sentence, add_special_tokens=False)

    # Truncate context to fit the sentence
    available = max_length - len(sentence_ids)
    if available < 0:
        # Sentence alone is too long
        sentence_ids = sentence_ids[-max_length:]
        context_ids = []
    else:
        # Keep last part of context
        context_ids = context_ids[-available:]

    # Build tensors
    input_ids = torch.tensor([context_ids + sentence_ids], dtype=torch.long)
    labels = input_ids.clone()
    labels[:, :len(context_ids)] = -100  # Ignore context in loss

    # Move to device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_ids = input_ids.to(device)
    labels = labels.to(device)

    # Compute loss
    with torch.no_grad():
        loss = model(input_ids=input_ids, labels=labels).loss

    return float(loss.detach().cpu())
def score_opt_contextual(sentence, context, tokenizer, model, max_length=1024):
    # Make SDPA use eager kernels (avoids half-only attention paths)
    os.environ.setdefault("PYTORCH_FORCE_SDP_EAGER", "1")
    
    # --- tokenize to ID lists (no specials) ---
    ctx_ids  = tokenizer(context,  add_special_tokens=False).input_ids
    sent_ids = tokenizer(sentence, add_special_tokens=False).input_ids
    
    # --- build final ids, truncating ONLY context so sentence stays intact ---
    if len(ctx_ids) + len(sent_ids) > max_length:
        keep_ctx = max(0, max_length - len(sent_ids))
        ctx_ids = ctx_ids[-keep_ctx:]
    final_ids = ctx_ids + sent_ids
    
    # tensorize once
    tokenize_input = torch.tensor([final_ids], dtype=torch.long)  # <— was encode(...)
    
    # Tokenize-only sentence length (already have it)
    sentence_len = len(sent_ids)
    
    # Create a mask aligned to the final ids
    mask = torch.cat([
        torch.zeros(len(final_ids) - sentence_len, dtype=torch.float32),
        torch.ones(sentence_len, dtype=torch.float32)
    ])
    
    # Device + force fp32 everywhere
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device).eval()
    model.float()                                # <— harden against fp16 overflow
    tensor_input = tokenize_input.to(device)
    mask = mask.to(device)
    
    # Single forward pass; disable autocast; keep logits in fp32
    with torch.no_grad():
        if torch.cuda.is_available():
            ctx = torch.cuda.amp.autocast(enabled=False)
        else:
            class _Noop:
                def __enter__(self): pass
                def __exit__(self, *a): pass
            ctx = _Noop()
        with ctx:
            logits = model(input_ids=tensor_input).logits.float()
    
    # Standard causal shift
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = tensor_input[..., 1:].contiguous()
    
    # Per-token CE (no reduction)
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction='none'
    ).view(shift_labels.size())
    
    # Align mask with prediction positions (drop first position)
    masked_loss = loss * mask[1:]
    masked_loss = masked_loss.sum() / mask.sum()   # avg over sentence tokens only
    
    # Return like your original (NumPy scalar)
    return masked_loss

In [48]:
ppl_contex = []
models = ['gpt2',
          'pythia']

for model_id in models:
    if model_id == 'gpt2':
        model = GPT2LMHeadModel.from_pretrained('gpt2')
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    elif model_id == 'pythia':
        MODEL_ID = "EleutherAI/pythia-160m-deduped"     # or "EleutherAI/pythia-70m
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, add_special_tokens=False)
        model     = AutoModelForCausalLM.from_pretrained(MODEL_ID)
        # Ensure pad token exists
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.pad_token_id
   
    for j, story_id  in enumerate(story_ids):
        story_segs = pd.read_excel('./A. script_prompting/story_segs.xlsx',
                           sheet_name='%s_segs' % story_id)['events'].to_list()
        story_segs = np.array(story_segs)[keep_ids[j]]
        ppl_story = []
        context = ''
        for seg in story_segs:
            if model_id=='pythia':
                perplexity = score_pythia_contextual(seg, context, tokenizer, model)
            else:
                perplexity = score_gpt_contextual(seg, context, tokenizer, model)
            context = context+seg
            ppl_story.append(perplexity)
        np.save(base_path+'/E. semantic_features/PPL_context_%s_%s' % (model_id,story_id), ppl_story)
        

C:\Users\hchen\AppData\Local\Temp\ipykernel_29236\1232327891.py:117: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx = torch.cuda.amp.autocast(enabled=False)


RuntimeError: value cannot be converted to type at::Half without overflow